# Community structure on the Ising J graph

Builds on `inverse_ising.ipynb`. The Ising J matrix is a signed weighted graph over Pokémon — `+J` edges are synergies (want to be on the same team), `−J` edges are competitions (want to *not* be on the same team). Pair-level analysis was already validated; the natural abstraction up is **groups**:

- **+J communities = archetype groups** — Pokémon that mutually synergize.
- **−J communities = role pools** — Pokémon that compete for the same slot.

We run Louvain modularity-maximization on each signed subgraph separately. Then cross-tabulate: mons sharing a +J community AND a −J community are intra-archetype substitutes (Pelipper/Politoed); mons sharing a −J community but in different +J communities are **cross-archetype substitutes** (Torkoal/Pelipper, if recoverable).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

import helpers

DATA_PATH = "gen9championsvgc2026regma-1760.json"
TEAM_SIZE = 6
EPS = 0.01
RNG_SEED = 0

chaos = helpers.load_chaos(DATA_PATH)
vocab = helpers.build_vocab(chaos, min_usage=0.002)
C = helpers.build_cooccurrence(chaos, vocab)
n = len(vocab)

m, p_joint = helpers.binary_moments(chaos, vocab, C, team_size=TEAM_SIZE)
Corr = helpers.binary_correlation(m, p_joint)
J, _ = helpers.ising_gaussian(Corr, eps=EPS)

iu, ju = np.triu_indices_from(J, k=1)
name_to_idx = {name: i for i, name in enumerate(vocab)}

print(f"metagame: {chaos.metagame}")
print(f"vocab:    {n}")
print(f"J range:  [{J.min():.3f}, {J.max():.3f}]")

## +J archetype communities

Build the positive-J subgraph (edges only where `J[i,j] > 0`, weighted by J magnitude). Run Louvain — it returns a partition that maximizes weighted modularity. Each community is a set of Pokémon that mutually synergize: a team archetype.

No threshold needed for +J — synergies are sparser than competitions, so the graph isn't pathologically dense.

In [ ]:
G_pos = nx.Graph()
G_pos.add_nodes_from(range(n))
for k in range(len(iu)):
    a, b = iu[k], ju[k]
    if J[a, b] > 0:
        G_pos.add_edge(a, b, weight=float(J[a, b]))

print(f"+J subgraph: {G_pos.number_of_nodes()} nodes, {G_pos.number_of_edges()} edges")

archetypes = sorted(
    nx.community.louvain_communities(G_pos, weight="weight", seed=RNG_SEED),
    key=len,
    reverse=True,
)
print(f"found {len(archetypes)} +J communities\n")

# Per-Pokémon archetype membership for later cross-tab
arch_of = np.full(n, -1, dtype=int)
for cid, c in enumerate(archetypes):
    for x in c:
        arch_of[x] = cid

for cid, c in enumerate(archetypes):
    top = sorted(c, key=lambda x: -m[x])[:10]
    print(f"archetype {cid:>2} (n={len(c):>2}): " + ", ".join(f"{vocab[x]} ({m[x]*100:.1f}%)" for x in top))

## −J role-pool communities

Build the negative-J subgraph (edges where `J[i,j] < 0`, weighted by `−J` magnitude). Threshold by quantile: the raw −J graph is nearly complete (every pair has *some* weak competition because team slots are finite), and Louvain on a near-complete graph collapses to one or two giant blobs. Thresholding to the top 5% of −J pair magnitudes lets community structure emerge.

**Expectation, calibrated.** The +J side worked cleanly. The −J side is harder for a structural reason: competition in team-building is broader than role-substitution. Two mons can mutually depress each other's probability of appearing for many reasons — same role, same archetype slot, same team-size budget pressure — and the Gaussian approximation pools all of those. Clean role pools should emerge for sharply-defined slots (Mega slot, Tera Type, designated weather setter); diffuse competition gets averaged in.

In [ ]:
neg_J_values = -J[iu, ju][J[iu, ju] < 0]
threshold = np.quantile(neg_J_values, 0.95)
print(f"-J threshold (95th percentile): {threshold:.4f}")

G_neg = nx.Graph()
G_neg.add_nodes_from(range(n))
for k in range(len(iu)):
    a, b = iu[k], ju[k]
    if -J[a, b] > threshold:
        G_neg.add_edge(a, b, weight=float(-J[a, b]))

isolated = [v for v, d in G_neg.degree() if d == 0]
print(f"-J subgraph (thresholded): {G_neg.number_of_nodes() - len(isolated)} connected nodes, {G_neg.number_of_edges()} edges, {len(isolated)} isolated\n")

role_pools = sorted(
    [c for c in nx.community.louvain_communities(G_neg, weight="weight", seed=RNG_SEED) if len(c) > 1],
    key=len,
    reverse=True,
)
print(f"found {len(role_pools)} -J communities (size > 1)\n")

pool_of = np.full(n, -1, dtype=int)
for cid, c in enumerate(role_pools):
    for x in c:
        pool_of[x] = cid

for cid, c in enumerate(role_pools):
    top = sorted(c, key=lambda x: -m[x])[:10]
    print(f"pool {cid:>2} (n={len(c):>2}): " + ", ".join(f"{vocab[x]} ({m[x]*100:.1f}%)" for x in top))

## Cross-tabulation: archetype × role pool

For each Pokémon we now have two community labels:

- archetype (+J community) — what kind of team it goes on
- role pool (−J community) — what other Pokémon compete with it for a slot

The contingency table reveals different structural cases:

- A **single (archetype, pool) cell** with many mons = intra-archetype substitutes (sand sweepers within the sand archetype).
- A **pool spanning multiple archetypes** = cross-archetype substitutes (Mega slot competitors across all archetypes; weather setters across weather archetypes).
- An **archetype spanning multiple pools** = internally-differentiated archetype (different roles within sand).

This is the structural answer to the role-vs-archetype question we kept hitting earlier.

In [ ]:
# Build contingency table: rows = archetypes, cols = role pools (+ "none" for unpooled)
n_arch = len(archetypes)
n_pool = len(role_pools)
contingency = np.zeros((n_arch, n_pool + 1), dtype=int)
for i in range(n):
    arch = arch_of[i]
    pool = pool_of[i] if pool_of[i] >= 0 else n_pool
    contingency[arch, pool] += 1

col_labels = [f"pool {i}" for i in range(n_pool)] + ["(none)"]
row_labels = [f"arch {i}" for i in range(n_arch)]

fig, ax = plt.subplots(figsize=(max(8, 1.5 * (n_pool + 1)), max(6, 0.5 * n_arch)))
im = ax.imshow(contingency, cmap="viridis", aspect="auto")
ax.set_xticks(range(n_pool + 1)); ax.set_xticklabels(col_labels, rotation=30, ha="right")
ax.set_yticks(range(n_arch)); ax.set_yticklabels(row_labels)
for i in range(n_arch):
    for j in range(n_pool + 1):
        v = contingency[i, j]
        if v > 0:
            ax.text(j, i, str(v), ha="center", va="center",
                    color="white" if v < contingency.max() / 2 else "black", fontsize=10)
ax.set_title("Archetype × role-pool contingency (cell value = # of Pokémon)")
plt.colorbar(im, ax=ax, label="count", shrink=0.7)
plt.tight_layout()
plt.show()

# Highlight cross-archetype pools: pools that span >1 archetype
print("Pools spanning multiple archetypes (cross-archetype substitute pools):")
for pool_id, c in enumerate(role_pools):
    arch_breakdown = {}
    for x in c:
        a = arch_of[x]
        arch_breakdown.setdefault(a, []).append(x)
    if len(arch_breakdown) > 1:
        print(f"\n  pool {pool_id}  (spans {len(arch_breakdown)} archetypes):")
        for a, members in sorted(arch_breakdown.items(), key=lambda kv: -len(kv[1])):
            top = sorted(members, key=lambda x: -m[x])[:5]
            print(f"    from arch {a}: {', '.join(vocab[x] for x in top)}")

## Visualization: the Ising J graph colored by archetype

Filtered network graph (top 3% of |J| pairs as in `inverse_ising.ipynb`), with nodes colored by their +J archetype membership. Blue solid edges = synergies, red dashed = competitions. Node size ∝ usage. Visually surfaces both within-cluster cohesion (+J edges inside each archetype color) and cross-cluster competition (−J edges between colors).

In [ ]:
J_abs = np.abs(J[iu, ju])
viz_thr = np.quantile(J_abs, 0.97)

G = nx.Graph()
for i in range(n):
    G.add_node(i, name=vocab[i], usage=m[i], arch=arch_of[i])
for k in range(len(iu)):
    a, b = iu[k], ju[k]
    if abs(J[a, b]) >= viz_thr:
        G.add_edge(a, b, weight=float(J[a, b]), abs_weight=float(abs(J[a, b])))

isolated = [i for i, d in G.degree() if d == 0]
G_drawn = G.copy()
G_drawn.remove_nodes_from(isolated)
print(f"viz graph: {G_drawn.number_of_nodes()} nodes, {G_drawn.number_of_edges()} edges")

pos = nx.spring_layout(G_drawn, weight="abs_weight", k=0.6, iterations=100, seed=RNG_SEED)

edges_pos = [(u, v) for u, v, d in G_drawn.edges(data=True) if d["weight"] > 0]
edges_neg = [(u, v) for u, v, d in G_drawn.edges(data=True) if d["weight"] < 0]
pos_widths = [3 * abs(G_drawn[u][v]["weight"]) for u, v in edges_pos]
neg_widths = [3 * abs(G_drawn[u][v]["weight"]) for u, v in edges_neg]

node_sizes = [80 + 4000 * G_drawn.nodes[i]["usage"] for i in G_drawn.nodes()]
node_colors = [G_drawn.nodes[i]["arch"] for i in G_drawn.nodes()]

fig, ax = plt.subplots(figsize=(16, 14))
nx.draw_networkx_edges(G_drawn, pos, edgelist=edges_pos, edge_color="#1f77b4",
                       alpha=0.4, width=pos_widths, ax=ax)
nx.draw_networkx_edges(G_drawn, pos, edgelist=edges_neg, edge_color="#d62728",
                       alpha=0.5, width=neg_widths, style="dashed", ax=ax)
nodes = nx.draw_networkx_nodes(G_drawn, pos, node_size=node_sizes,
                               node_color=node_colors, cmap=plt.cm.tab10, alpha=0.9, ax=ax)

top_usage = sorted(G_drawn.nodes(), key=lambda i: -G_drawn.nodes[i]["usage"])[:30]
labels = {i: vocab[i] for i in top_usage}
nx.draw_networkx_labels(G_drawn, pos, labels=labels, font_size=9, ax=ax)

ax.set_title(f"Ising J graph, nodes colored by +J archetype  (|J| ≥ {viz_thr:.3f}; "
             f"blue = +J, red dashed = -J)")
ax.axis("off")
plt.tight_layout()
plt.show()